# 05 - Phase 5: inference, evaluation and the final predictions

> Run with **Save Version -> Save & Run All (Commit)**. Settings: **Accelerator = GPU T4 x2**, **Internet = On**.
> This is the **inference notebook**: it loads the trained LoRA adapters and needs no training.

**Inputs (Add Input):**
1. the Phase 0 dataset(s) holding `train_processed.csv` **and** `test_processed.csv` (the unseen 1,000 questions),
2. the Phase 2 dataset `nlp-qna-retrieval` (`question_neighbours.json`),
3. the adapters dataset `nlp-qna-adapters` (`nn/` and `none/`).

**What it does**
1. **Exact-match router** - an unseen question whose text matches a training question exactly gets that question's stored (medoid) answer. No model call.
2. Every other unseen question is answered by the fine-tuned model with its 2 nearest training Q&A pairs in the prompt.
3. **Hold-out evaluation** on questions the model never saw (a grouped split, so no question text crosses train/test): four systems on the same questions -
   B0a nearest-neighbour answer, **B1** zero-shot base model, **B2** fine-tuned without retrieval, **proposed** fine-tuned with retrieval - plus a bootstrap confidence interval.
4. The router is scored separately, because it only applies to repeated questions. The combined number for the unseen set is printed as a clearly labelled *estimate*.
5. Writes `xt_predictions.csv` and `metrics.json`.

Generation runs one worker per GPU and **appends every finished batch to disk**: if the session dies, re-run and it resumes.

In [ ]:
!pip install -q -U peft bitsandbytes accelerate huggingface_hub rouge-score sacrebleu

In [ ]:
%%writefile prompts.py
"""Phase 4: build the training / inference prompts.

Modes
  none : question only                                   (baseline B2: plain fine-tuning)
  nn   : question + the 2 nearest training Q&A pairs      (proposed model without PDFs)
  full : nn + source-page excerpts                         (proposed model with PDFs)

Neighbours for TRAIN questions already exclude the question's own group (see Phase 2), so the model is trained to
synthesise an answer from related-but-different answers, not to copy an identical one.
Pure ASCII source on purpose.
"""
import json
import random

SYSTEM = (
    "You are an expert Indian agricultural advisor. Answer the farmer's question in English with a detailed, "
    "well-structured advisory: key facts or symptoms, thresholds where relevant, cultural, biological and chemical "
    "management with exact doses and units, and safety notes. Use the reference material when it applies to this "
    "question; never invent doses."
)


def trim_words(text, n):
    w = text.split()
    return " ".join(w[:n]) + (" ..." if len(w) > n else "")


def trim_tokens(text, n, tok):
    """Cut text to n tokens. Token budgets (not word counts) keep every prompt inside the length limit: numeric-heavy
    text can use twice as many tokens per word as prose."""
    ids = tok(text, add_special_tokens=False)["input_ids"]
    return text if len(ids) <= n else tok.decode(ids[:n]) + " ..."


def build_user(mode, question, refs, excerpts, ref_words=300, pdf_words=180, tok=None, ref_tokens=380, pdf_tokens=200):
    """refs: list of (question, answer); excerpts: list of (label, text). With tok given, trimming is by tokens."""
    def cut(text, words, tokens):
        return trim_tokens(text, tokens, tok) if tok is not None else trim_words(text, words)
    if mode == "none":
        return "Question: " + question
    parts = ["Reference answers to similar questions:"]
    for i, (q, a) in enumerate(refs, 1):
        parts.append("[%d] Q: %s\nA: %s" % (i, q, cut(a, ref_words, ref_tokens)))
    if mode == "full" and excerpts:
        parts.append("Source excerpts:")
        for label, t in excerpts:
            parts.append("[%s] %s" % (label, cut(t, pdf_words, pdf_tokens)))
    parts.append("Question: " + question)
    return "\n\n".join(parts)


class Context:
    """Holds the lookups needed to build prompts for any query set."""

    def __init__(self, rows, page_text=None, chunk_text=None):
        self.qid2row = {r["question_id"]: r for r in rows}
        self.rows = rows
        self.page_text = page_text or {}      # (doc_id, page) -> first chunk text of that page
        self.chunk_text = chunk_text or []    # chunk id -> text
        self._cit = {}

    def citations(self, qid):
        if qid not in self._cit:
            self._cit[qid] = json.loads(self.qid2row[qid]["citations_json"])
        return self._cit[qid]

    def propagated_pages(self, neighbour_ids, limit=2):
        """Pages cited by the nearest training questions, in neighbour order. Returns [(label, text)]."""
        out, seen = [], set()
        for nid in neighbour_ids:
            for c in self.citations(nid):
                if c.get("kind") != "zoho":
                    continue
                for p in c.get("pages", []):
                    key = (c["doc_id"], p)
                    if key in self.page_text and key not in seen:
                        seen.add(key)
                        name = (c.get("name") or "source")[:60]
                        out.append(("%s, p.%d" % (name, p), self.page_text[key]))
                        if len(out) == limit:
                            return out
        return out


def make_example(ctx, mode, qid, question, neighbours, chunk_hits=None, answer=None, noise=0.0, rng=None,
                 pool_qids=None, n_refs=2, tok=None):
    """neighbours: [[question_id, score], ...] best first."""
    rng = rng or random.Random(0)
    nids = [n[0] for n in neighbours]
    refs_ids = nids[:n_refs]
    if noise and pool_qids and len(refs_ids) == n_refs and rng.random() < noise:
        refs_ids[-1] = rng.choice(pool_qids)      # RAFT-style distractor: the model learns to ignore bad context
    refs = [(ctx.qid2row[i]["question"], ctx.qid2row[i]["final_answer"]) for i in refs_ids]
    excerpts = []
    if mode == "full":
        excerpts = ctx.propagated_pages(nids[:3])
        if len(excerpts) < 2 and chunk_hits and ctx.chunk_text:
            for cid in chunk_hits:
                if len(excerpts) == 2:
                    break
                excerpts.append(("retrieved passage", ctx.chunk_text[cid]))
        if noise and excerpts and pool_qids and rng.random() < noise:
            other = ctx.propagated_pages([rng.choice(pool_qids)], limit=1)
            if other:
                excerpts[-1] = other[0]
    ex = {"qid": qid, "mode": mode, "system": SYSTEM, "user": build_user(mode, question, refs, excerpts, tok=tok)}
    if answer is not None:
        ex["answer"] = answer
    return ex

In [ ]:
%%writefile eval_harness.py
"""Single evaluation harness used by every system (baselines, ablations, final model).

    from eval_harness import evaluate
    scores = evaluate(preds, refs)            # dicts {question_id: text}
    scores = evaluate(preds, refs, bertscore=True, meteor=True)   # slower extras

Metrics: ROUGE-1/2/L F1, corpus BLEU (sacrebleu), length ratio, agronomic fact recall/precision/F1
(number+unit pairs, formulation-tagged chemicals such as "dinotefuran 20 SG", binomials), and optionally
METEOR (nltk) and BERTScore-F1. Extras are lazy-imported so the core runs with rouge-score + sacrebleu only.
"""
import re
import statistics as st

from rouge_score import rouge_scorer

_UNIT = {"gm": "g", "gram": "g", "grams": "g", "gms": "g", "litre": "l", "litres": "l", "lit": "l", "liter": "l",
         "liters": "l", "hectare": "ha", "hectares": "ha", "per cent": "%", "percent": "%", "quintals": "q",
         "quintal": "q", "tonnes": "t", "tonne": "t", "day": "days", "kgs": "kg"}
NUM_UNIT_RE = re.compile(
    r"(\d+(?:\.\d+)?)\s*(?:-|\u2013|to)?\s*(\d+(?:\.\d+)?)?\s*"
    r"(kg|kgs|gms?|grams?|g|mg|ml|litres?|liters?|lit|l|quintals?|q|tonnes?|t|ha|hectares?|acre|cm|mm|"
    r"days?|per\s*cent|percent|%|ppm)\b"
    r"(?:\s*(?:/|per)\s*(ha|hectare|acre|litres?|liters?|lit|l|plant|kg|m2|10\s*l))?", re.I)
CHEM_RE = re.compile(r"\b([A-Za-z][A-Za-z\-]{3,})\s+\d+(?:\.\d+)?\s*%?\s*"
                     r"(SG|SL|EC|WP|WG|SC|FS|DP|GR|ZC|OD|EW|CS|ME|WS|DF)\b")
BINOM_RE = re.compile(r"\(([A-Z][a-z]{3,}\s+[a-z]{4,})\)")


def _u(x):
    x = re.sub(r"\s+", " ", x.lower().strip())
    return _UNIT.get(x, x)


def extract_facts(text):
    facts = set()
    for m in NUM_UNIT_RE.finditer(text):
        a, b, unit, per = m.groups()
        facts.add("%s%s%s%s" % (a, "-" + b if b else "", _u(unit), "/" + _u(per) if per else ""))
    for m in CHEM_RE.finditer(text):
        facts.add("chem:" + m.group(1).lower())
    for m in BINOM_RE.finditer(text):
        facts.add("sp:" + m.group(1).lower())
    return facts


def _fact_scores(preds, refs):
    r_list, p_list, f_list = [], [], []
    for p, r in zip(preds, refs):
        fr, fp = extract_facts(r), extract_facts(p)
        if not fr:
            continue
        inter = len(fr & fp)
        rec = inter / len(fr)
        prec = inter / len(fp) if fp else 0.0
        r_list.append(rec)
        p_list.append(prec)
        f_list.append(0.0 if rec + prec == 0 else 2 * rec * prec / (rec + prec))
    n = max(1, len(r_list))
    return {"fact_recall": sum(r_list) / n, "fact_precision": sum(p_list) / n, "fact_f1": sum(f_list) / n,
            "fact_n_refs": len(r_list)}


def evaluate(preds, refs, bertscore=False, meteor=False, bert_model="roberta-large"):
    ids = [i for i in refs if i in preds]
    P = [preds[i] or "" for i in ids]
    R = [refs[i] for i in ids]
    sc = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    agg = {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}
    for p, r in zip(P, R):
        s = sc.score(r, p)
        for k in agg:
            agg[k] += s[k].fmeasure
    n = max(1, len(ids))
    out = {"n": len(ids), "n_missing": len(refs) - len(ids)}
    out.update({k: v / n for k, v in agg.items()})
    import sacrebleu
    out["bleu"] = sacrebleu.corpus_bleu(P, [R]).score / 100.0
    pl, rl = [len(p.split()) for p in P], [len(r.split()) for r in R]
    out["len_ratio_mean"] = (sum(pl) / n) / max(1e-9, sum(rl) / n)
    out["len_ratio_median_per_item"] = st.median([a / max(1, b) for a, b in zip(pl, rl)]) if ids else 0.0
    out.update(_fact_scores(P, R))
    if meteor:
        from nltk.translate.meteor_score import meteor_score
        out["meteor"] = sum(meteor_score([r.split()], p.split()) for p, r in zip(P, R)) / n
    if bertscore:
        from bert_score import score as bs
        _, _, f1 = bs(P, R, model_type=bert_model, lang="en", batch_size=16, verbose=False)
        out["bertscore_f1"] = float(f1.mean())
    return out


def evaluate_by_group(preds, refs, groups, **kw):
    """groups: {name: set(question_ids)} -> {name: scores}; e.g. novel vs repeated, or per crop."""
    return {g: evaluate(preds, {i: refs[i] for i in ids if i in refs}, **kw) for g, ids in groups.items()}


if __name__ == "__main__":
    t = "Spray dinotefuran 20 SG @ 60 g/acre; seed rate 25 kg/ha. Pest: Amrasca devastans (Amrasca devastans)."
    print(extract_facts(t))
    print(evaluate({"a": t}, {"a": t}))

In [ ]:
%%writefile infer.py
"""Phase 5 generation worker: one process per GPU.

Reads a jobs file (jsonl: uid, adapter, prio, system, user), generates greedy answers in batches, and appends every
finished batch to --out immediately, so a crashed or time-limited run resumes where it stopped (done uids are skipped).
adapter is "nn" / "none" (a LoRA folder under --adapter_dir) or "base" (the same model with the adapter switched off:
the zero-shot baseline B1).  Pure ASCII source on purpose.

    CUDA_VISIBLE_DEVICES=0 python infer.py --jobs jobs.jsonl --adapter_dir /path --adapters nn --out gen_gpu0.jsonl
"""
import argparse
import json
import os
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, LogitsProcessor, LogitsProcessorList
from peft import PeftModel


def parse():
    p = argparse.ArgumentParser()
    p.add_argument("--jobs", required=True)
    p.add_argument("--adapter_dir", required=True)
    p.add_argument("--adapters", required=True, help="comma list handled by this worker, e.g. none,base")
    p.add_argument("--out", required=True)
    p.add_argument("--model", default="Qwen/Qwen2.5-3B-Instruct")
    p.add_argument("--max_new", type=int, default=768)
    p.add_argument("--batch", type=int, default=12)
    p.add_argument("--rep_penalty", type=float, default=1.05, help="HF repetition penalty (acts on prompt AND answer tokens)")
    p.add_argument("--gen_rep_penalty", type=float, default=1.0, help="penalty on tokens of the ANSWER only (1.0 = off)")
    p.add_argument("--gen_no_repeat", type=int, default=0, help="forbid repeating any n-gram of the ANSWER (0 = off)")
    p.add_argument("--do_sample", action="store_true")
    p.add_argument("--temperature", type=float, default=0.7)
    p.add_argument("--top_p", type=float, default=0.9)
    p.add_argument("--seed", type=int, default=1234)
    p.add_argument("--hours", type=float, default=10.0)
    p.add_argument("--no_quant", action="store_true")
    return p.parse_args()


class GenRepetitionPenalty(LogitsProcessor):
    """Repetition penalty over the generated tokens only (the HF one also penalises every prompt token, which
    discourages copying wording from the reference answers in the prompt). Assumes left padding."""

    def __init__(self, penalty, prompt_len):
        self.p, self.plen = penalty, prompt_len

    def __call__(self, input_ids, scores):
        g = input_ids[:, self.plen:]
        if g.shape[1] == 0:
            return scores
        s = torch.gather(scores, 1, g)
        s = torch.where(s < 0, s * self.p, s / self.p)
        return scores.scatter(1, g, s)


class GenNoRepeatNGram(LogitsProcessor):
    """Forbid any n-gram that already occurs in the generated answer (prompt excluded). Incremental per row."""

    def __init__(self, n, prompt_len):
        self.n, self.plen, self.seen = n, prompt_len, None

    def __call__(self, input_ids, scores):
        B, L = input_ids.shape
        if self.seen is None:
            self.seen = [dict() for _ in range(B)]
        glen = L - self.plen
        if glen < self.n - 1:
            return scores
        rows = input_ids[:, self.plen:].tolist()
        for b in range(B):
            g = rows[b]
            if glen >= self.n:                      # register the newest complete n-gram
                self.seen[b].setdefault(tuple(g[-self.n:-1]), set()).add(g[-1])
            banned = self.seen[b].get(tuple(g[len(g) - self.n + 1:]))
            if banned:
                scores[b, list(banned)] = -float("inf")
        return scores


def load_model(a, names):
    cuda = torch.cuda.is_available()
    if a.no_quant:
        base = AutoModelForCausalLM.from_pretrained(a.model, torch_dtype=torch.float16 if cuda else torch.float32)
        if cuda:
            base = base.to("cuda:0")
    else:
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
                                 bnb_4bit_use_double_quant=True)
        base = AutoModelForCausalLM.from_pretrained(a.model, quantization_config=bnb, torch_dtype=torch.float16,
                                                    device_map={"": 0})
    real = [n for n in names if n != "base"]
    if not real:
        return base.eval(), False
    model = PeftModel.from_pretrained(base, os.path.join(a.adapter_dir, real[0]), adapter_name=real[0])
    for n in real[1:]:
        model.load_adapter(os.path.join(a.adapter_dir, n), adapter_name=n)
    return model.eval(), True


def generate(model, tok, texts, a, device):
    enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(device)
    plen = enc["input_ids"].shape[1]
    procs = LogitsProcessorList()
    if getattr(a, "gen_rep_penalty", 1.0) != 1.0:
        procs.append(GenRepetitionPenalty(a.gen_rep_penalty, plen))
    if getattr(a, "gen_no_repeat", 0):
        procs.append(GenNoRepeatNGram(a.gen_no_repeat, plen))
    kw = dict(max_new_tokens=a.max_new, repetition_penalty=a.rep_penalty, pad_token_id=tok.pad_token_id,
              logits_processor=procs)
    if getattr(a, "do_sample", False):
        torch.manual_seed(a.seed)
        kw.update(do_sample=True, temperature=a.temperature, top_p=a.top_p, top_k=0)
    else:
        kw.update(do_sample=False, temperature=None, top_p=None, top_k=None)
    with torch.no_grad():
        out = model.generate(**enc, **kw)
    gen = out[:, enc["input_ids"].shape[1]:]
    res = []
    for g in gen:
        ids = g.tolist()
        n = len(ids)
        while n and ids[n - 1] == tok.pad_token_id:
            n -= 1
        res.append((tok.decode(ids[:n], skip_special_tokens=True).strip(), n))
    return res


def main():
    a = parse()
    names = a.adapters.split(",")
    jobs = [json.loads(l) for l in open(a.jobs, encoding="utf-8")]
    jobs = [j for j in jobs if j["adapter"] in names]
    done = set()
    if os.path.exists(a.out):
        done = {json.loads(l)["uid"] for l in open(a.out, encoding="utf-8") if l.strip()}
    todo = [j for j in jobs if j["uid"] not in done]
    print("jobs for %s: %d | already done: %d | to do: %d" % (names, len(jobs), len(done), len(todo)), flush=True)
    if not todo:
        return
    tok = AutoTokenizer.from_pretrained(a.model)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    for j in todo:
        j["text"] = tok.apply_chat_template([{"role": "system", "content": j["system"]}, {"role": "user", "content": j["user"]}],
                                            tokenize=False, add_generation_prompt=True)
        j["plen"] = len(tok(j["text"], add_special_tokens=False)["input_ids"])
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    model, has_peft = load_model(a, names)
    t_end = time.time() + a.hours * 3600
    t0 = time.time()
    n_done = 0
    fout = open(a.out, "a", encoding="utf-8")
    for name in names:
        group = sorted((j for j in todo if j["adapter"] == name), key=lambda j: (j["prio"], -j["plen"]))
        if not group:
            continue
        if name != "base" and has_peft:
            model.set_adapter(name)
        print("adapter %s: %d jobs" % (name, len(group)), flush=True)
        i = 0
        bs = a.batch
        while i < len(group):
            if time.time() > t_end:
                print("time limit reached - stopping; rerun to resume", flush=True)
                return
            batch = group[i:i + bs]
            try:
                if name == "base" and has_peft:
                    with model.disable_adapter():
                        res = generate(model, tok, [j["text"] for j in batch], a, device)
                else:
                    res = generate(model, tok, [j["text"] for j in batch], a, device)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if bs == 1:
                    raise
                bs = max(1, bs // 2)
                print("out of memory - batch size now", bs, flush=True)
                continue
            for j, (text, n) in zip(batch, res):
                fout.write(json.dumps({"uid": j["uid"], "adapter": name, "text": text, "n_tokens": n,
                                       "hit_limit": n >= a.max_new}, ensure_ascii=False) + "\n")
            fout.flush()
            i += len(batch)
            n_done += len(batch)
            el = time.time() - t0
            print("[%5.0f min] %s %d/%d | %.1f s per answer" % (el / 60, name, i, len(group), el / n_done), flush=True)
    print("DONE", n_done, "answers in %.1f min" % ((time.time() - t0) / 60), flush=True)


if __name__ == "__main__":
    main()

## 0. Configuration

In [ ]:
import glob, os, sys, json, gzip, csv, random, time, subprocess, collections as C
csv.field_size_limit(10 ** 9)
_missing = [f for f in ("prompts.py", "eval_harness.py", "infer.py") if not os.path.exists(f)]
assert not _missing, "missing " + str(_missing) + " - run the %%writefile cells at the top first (use Run All, do not start from a lower cell)"
sys.path.insert(0, os.getcwd())

import torch
if not os.environ.get("P5_CPU"):   # (CPU smoke-test hook, never set on Kaggle)
    assert torch.cuda.is_available(), ("No GPU attached to this session. Right panel -> Session options -> Accelerator -> "
                                       "GPU T4 x2 (the session restarts), then Run All from the top.")
    print("GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

MODEL = os.environ.get("P5_MODEL", "Qwen/Qwen2.5-3B-Instruct")
N_HOLDOUT = 300            # hold-out questions scored (same ones for every system). 1500 = the whole hold-out set (about 5x slower)
UNSEEN_ADAPTER = "nn"      # which adapter answers the non-repeated unseen questions: "nn" (with retrieval) or "none"
MAX_NEW = 768              # generation limit in tokens (the median reference answer is about 417 words ~ 600 tokens)
BATCH = 12
GEN_HOURS = 9.0            # each worker stops cleanly after this long; re-run to resume
BERTSCORE = False          # True adds BERTScore (downloads roberta-large; slower)
WORK = "/kaggle/working"

def find(pattern, required=True):
    hit = glob.glob("/kaggle/input/**/" + pattern, recursive=True)
    if not hit and required:
        print("not found:", pattern, "- files under /kaggle/input (first 40):")
        for i, (dp, dn, fn) in enumerate(os.walk("/kaggle/input")):
            for x in fn[:8]:
                print("  ", os.path.join(dp, x))
            if i > 40:
                break
        raise AssertionError("attach a dataset containing " + pattern)
    return hit[0] if hit else None

def op(path):
    return gzip.open(path, "rt", encoding="utf-8") if path.endswith(".gz") else open(path, encoding="utf-8")

TRAIN_CSV, UNSEEN_CSV, NN_PATH = find("train_processed*.csv"), find("test_processed*.csv"), find("question_neighbours.json*")
AD_DIR = os.path.dirname(os.path.dirname(find("nn/adapter_config.json")))
assert os.path.exists(AD_DIR + "/none/adapter_config.json"), "the adapters dataset must contain nn/ and none/ folders"
print("adapters:", AD_DIR, "| model:", MODEL, "| hold-out questions:", N_HOLDOUT, "| unseen adapter:", UNSEEN_ADAPTER)

## 1. Router and prompts (CPU)

In [ ]:
import prompts as P
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL)
rows = list(csv.DictReader(open(TRAIN_CSV, encoding="utf-8")))
unseen = list(csv.DictReader(open(UNSEEN_CSV, encoding="utf-8")))
NN = json.load(op(NN_PATH))
ctx = P.Context(rows)
qid2row = ctx.qid2row

# exact-match router: unseen question text == a training question -> that group's medoid answer
medoid = {r["group_id"]: r for r in rows if r["is_medoid"] == "1"}
routed = {u["question_id"]: medoid[u["matched_group_id"]]["final_answer"]
          for u in unseen if u["in_train"] == "1" and u["matched_group_id"] in medoid}
gen_unseen = [u for u in unseen if u["question_id"] not in routed]
print("unseen questions: %d | routed by exact match: %d (%.1f%%) | to generate: %d" % (len(unseen), len(routed), 100 * len(routed) / len(unseen), len(gen_unseen)))

test_rows = [r for r in rows if r["split"] == "test"]
hold = random.Random(42).sample(test_rows, min(N_HOLDOUT, len(test_rows))) if N_HOLDOUT else test_rows
print("hold-out rows (never seen in training):", len(hold), "of", len(test_rows))

def mk(adapter, prio, tag, qid, question, mode, split):
    ex = P.make_example(ctx, mode, qid, question, NN[split].get(qid, []) if mode != "none" else [], tok=tok)
    return {"uid": "%s|%s|%s" % (adapter, tag, qid), "adapter": adapter, "prio": prio, "system": ex["system"], "user": ex["user"]}

jobs = []
for r in hold:
    q = r["question_id"]
    jobs.append(mk("nn", 0, "hold", q, r["question"], "nn", "test"))
    jobs.append(mk("none", 0, "hold", q, r["question"], "none", "test"))
    jobs.append(mk("base", 0, "hold", q, r["question"], "none", "test"))   # B1: same prompt as B2, adapter switched off
for u in gen_unseen:
    jobs.append(mk(UNSEEN_ADAPTER, 1, "xt", u["question_id"], u["question"], "nn" if UNSEEN_ADAPTER == "nn" else "none", "xt"))
os.makedirs(WORK + "/gen", exist_ok=True)
with open(WORK + "/jobs.jsonl", "w", encoding="utf-8") as f:
    for j in jobs:
        f.write(json.dumps(j, ensure_ascii=False) + "\n")
lens = [len(tok(j["system"] + j["user"], add_special_tokens=False)["input_ids"]) for j in jobs]
print("jobs:", len(jobs), dict(C.Counter(j["adapter"] for j in jobs)), "| prompt tokens mean %.0f max %d" % (sum(lens) / len(lens), max(lens)))

## 2. Generate - one worker per GPU (resumable)
The first batches are the longest prompts, so a memory problem shows up immediately (the worker halves its batch size by itself).

In [ ]:
if not os.environ.get("P5_CPU"):
    from huggingface_hub import snapshot_download
    snapshot_download(MODEL)
N_GPU = max(1, torch.cuda.device_count())
workers = [["nn"], ["none", "base"]] if N_GPU >= 2 else [["nn", "none", "base"]]
procs = []
for gpu, names in enumerate(workers):
    log = open("%s/gen/worker%d.log" % (WORK, gpu), "a")
    cmd = [sys.executable, "infer.py", "--jobs", WORK + "/jobs.jsonl", "--adapter_dir", AD_DIR, "--adapters", ",".join(names),
           "--out", "%s/gen/gen_gpu%d.jsonl" % (WORK, gpu), "--model", MODEL, "--max_new", str(MAX_NEW), "--batch", str(BATCH), "--hours", str(GEN_HOURS)]
    if os.environ.get("P5_CPU"):
        cmd.append("--no_quant")
    procs.append((gpu, subprocess.Popen(cmd, env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), TOKENIZERS_PARALLELISM="false"), stdout=log, stderr=subprocess.STDOUT)))
    print("started worker", gpu, "adapters", names)
t0 = time.time()
while any(p.poll() is None for _, p in procs):
    time.sleep(120)
    for gpu, p in procs:
        lines = open("%s/gen/worker%d.log" % (WORK, gpu), errors="ignore").read().splitlines()
        print("[%4.0f min] worker %d: %s" % ((time.time() - t0) / 60, gpu, lines[-1][:150] if lines else "(no output yet)"), flush=True)
for gpu, p in procs:
    print("worker", gpu, "exit code", p.returncode)
    if p.returncode:
        print(open("%s/gen/worker%d.log" % (WORK, gpu), errors="ignore").read()[-2000:])

## 3. Collect the answers

In [ ]:
gen = {}
for f in glob.glob(WORK + "/gen/gen_gpu*.jsonl"):
    for l in open(f, encoding="utf-8"):
        if l.strip():
            d = json.loads(l)
            gen[d["uid"]] = d
missing = [j["uid"] for j in jobs if j["uid"] not in gen]
print("generated: %d of %d jobs | missing: %d" % (len(gen), len(jobs), len(missing)))
if missing:
    print("WARNING: some jobs are missing (a worker stopped early). Run this notebook again - it resumes. First missing:", missing[:3])
cut = sum(d["hit_limit"] for d in gen.values())
print("answers cut off at MAX_NEW=%d tokens: %d (%.1f%%)" % (MAX_NEW, cut, 100 * cut / max(1, len(gen))))

## 4. Hold-out evaluation - the main results table for the report

In [ ]:
import numpy as np, pandas as pd
import eval_harness as H
def nearest(split, q):
    nb = NN[split].get(q)
    return qid2row[nb[0][0]]["final_answer"] if nb else ""

ids = [r["question_id"] for r in hold if all(("%s|hold|%s" % (a, r["question_id"])) in gen for a in ("nn", "none", "base"))]
refs = {r["question_id"]: r["final_answer"] for r in hold if r["question_id"] in set(ids)}
systems = {"B0a nearest-neighbour answer (no model)": {q: nearest("test", q) for q in ids}}
empty = {}
for name, ad in (("B1 zero-shot base model", "base"), ("B2 fine-tuned, no retrieval", "none"), ("PROPOSED fine-tuned + retrieval", "nn")):
    p, e = {}, 0
    for q in ids:
        t = gen["%s|hold|%s" % (ad, q)]["text"]
        if not t.strip():
            t, e = nearest("test", q), e + 1   # an empty generation falls back to the nearest answer (counted)
        p[q] = t
    systems[name] = p
    empty[name] = e
print("scored questions:", len(ids), "| empty generations replaced by the fallback:", empty)
res = {n: H.evaluate(p, refs, bertscore=BERTSCORE) for n, p in systems.items()}
cols = ["rouge1", "rouge2", "rougeL", "bleu", "len_ratio_mean", "fact_f1"] + (["bertscore_f1"] if BERTSCORE else [])
table = pd.DataFrame({n: {c: r[c] for c in cols} for n, r in res.items()}).T
pd.set_option("display.width", 200)
print(table.round(4).to_string())

# paired bootstrap on ROUGE-L (resample questions, 2000 times)
from rouge_score import rouge_scorer
sc = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
per = {n: np.array([sc.score(refs[q], p[q])["rougeL"].fmeasure for q in ids]) for n, p in systems.items()}
idx = np.random.default_rng(0).integers(0, len(ids), (2000, len(ids)))
def diff_ci(a, b):
    d = (per[a] - per[b])[idx].mean(axis=1)
    return "%+.4f  [95%% CI %+.4f, %+.4f]" % (per[a].mean() - per[b].mean(), np.percentile(d, 2.5), np.percentile(d, 97.5))
names = list(systems)
print("\nROUGE-L difference (paired bootstrap):")
print("  proposed - B2 (value of retrieval):", diff_ci(names[3], names[2]))
print("  proposed - B1 (value of fine-tuning):", diff_ci(names[3], names[1]))
print("  proposed - B0a (value of generating):", diff_ci(names[3], names[0]))

## 5. The router, scored separately
Only repeated questions can be routed, so this is measured on training questions that appear more than once: the stored answer is compared with a *different* human answer to the same question.

In [ ]:
groups = C.defaultdict(list)
for r in rows:
    groups[r["group_id"]].append(r)
rep = [g for g in groups.values() if len(g) >= 2]
rr = random.Random(7)
held, p_med, p_rand = {}, {}, {}
for g in rr.sample(rep, min(1500, len(rep))):
    med = next((x for x in g if x["is_medoid"] == "1"), g[0])
    h = rr.choice([x for x in g if x is not med])            # a human answer that the stored one is compared with
    held[h["question_id"]] = h["final_answer"]
    p_med[h["question_id"]] = med["final_answer"]            # the router's answer: the stored medoid
    p_rand[h["question_id"]] = rr.choice([x for x in g if x is not h])["final_answer"]
router_res = H.evaluate(p_med, held)
rand_res = H.evaluate(p_rand, held)
print("repeated-question groups: %d (of %d groups) | scored: %d" % (len(rep), len(groups), len(held)))
print("router (stored medoid answer)  : rougeL %.4f  rouge1 %.4f  bleu %.4f  fact_f1 %.4f" % tuple(router_res[k] for k in ("rougeL", "rouge1", "bleu", "fact_f1")))
print("random other human answer      : rougeL %.4f  rouge1 %.4f  bleu %.4f  fact_f1 %.4f" % tuple(rand_res[k] for k in ("rougeL", "rouge1", "bleu", "fact_f1")))
share = len(routed) / len(unseen)
prop = res[names[3]]
print("\nESTIMATE for the unseen set (not a measurement): %.1f%% routed x router score + %.1f%% generated x hold-out score of the proposed model" % (100 * share, 100 * (1 - share)))
for k in ("rougeL", "rouge1", "fact_f1"):
    print("   %-8s %.4f" % (k, share * router_res[k] + (1 - share) * prop[k]))

## 6. Write the predictions for the unseen set

In [ ]:
final, source, fallback = {}, {}, 0
for u in unseen:
    q = u["question_id"]
    if q in routed:
        final[q], source[q] = routed[q], "router"
        continue
    d = gen.get("%s|xt|%s" % (UNSEEN_ADAPTER, q))
    if d and d["text"].strip():
        final[q], source[q] = d["text"], "generated"
    else:
        final[q], source[q] = nearest("xt", q), "fallback_nearest"
        fallback += 1
assert len(final) == len(unseen) == 1000 and all(v.strip() for v in final.values()), "predictions incomplete"
with open(WORK + "/xt_predictions.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["question_id", "question", "final_answer"])
    for u in unseen:
        w.writerow([u["question_id"], u["question"], final[u["question_id"]]])
with open(WORK + "/xt_predictions_detailed.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["question_id", "source", "final_answer"])
    for u in unseen:
        w.writerow([u["question_id"], source[u["question_id"]], final[u["question_id"]]])
print("wrote xt_predictions.csv:", len(final), "rows |", dict(C.Counter(source.values())))
if fallback:
    print("WARNING: %d unseen questions used the nearest-answer fallback (missing generations) - re-run to resume." % fallback)
json.dump({"holdout_n": len(ids), "systems": res, "router": router_res, "router_random_other": rand_res,
           "routed_share": share, "bootstrap_rougeL": {"proposed_minus_B2": diff_ci(names[3], names[2]), "proposed_minus_B1": diff_ci(names[3], names[1]),
           "proposed_minus_B0a": diff_ci(names[3], names[0])}, "config": {"N_HOLDOUT": N_HOLDOUT, "MAX_NEW": MAX_NEW, "UNSEEN_ADAPTER": UNSEEN_ADAPTER, "MODEL": MODEL}},
          open(WORK + "/metrics.json", "w"), indent=1, default=float)

## 7. Look at a few answers (for the report's qualitative section)

In [ ]:
for q in ids[:3]:
    print("=" * 90, "\nQ:", qid2row[q]["question"][:200])
    print("--- reference:", refs[q][:450].replace("\n", " "))
    print("--- proposed :", systems[names[3]][q][:450].replace("\n", " "))
    print("--- B2       :", systems[names[2]][q][:300].replace("\n", " "))

## After it finishes
Output tab: `xt_predictions.csv` (the deliverable), `xt_predictions_detailed.csv` (marks each answer as router / generated), `metrics.json`, `gen/` (raw generations).
Send me the **tables printed in sections 4 and 5**.